In [1]:
from dotenv import load_dotenv
load_dotenv('env')

True

In [2]:
import os
from unsloth import FastLanguageModel
import torch
import wandb
from trl import DataCollatorForCompletionOnlyLM

from trl import SFTTrainer
from transformers import TrainingArguments, AutoTokenizer
from unsloth import is_bfloat16_supported

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [3]:
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # New Google 6 trillion tokens model 2.5x faster!
    "unsloth/gemma-2b-bnb-4bit",
]
MODEL_ID = "unsloth/gemma-7b-bnb-4bit"
FT_MODEL_NAME = f"AIMO-SFT-Unsloth-{MODEL_ID.split('/')[-1]}-AIMO-Tokenizer"

## MODEL TRAINING VARIABLE
MAX_LENGTH = 4096
MAX_PROMPT_LENGTH = 1024
BATCH_SIZE = 2
LOAD_IN_4BIT = True
DTYPE = None

## LORA SETTINGS
LORA_R = 128
LORA_ALPHA = 64
LORA_DROPOUT = 0
LORA_BIAS = 'none'
USE_GRADIENT_CHECKPOINTING = "unsloth" # @param ["True", "False", "unsloth"] # 'unsloth' # True or 'unsloth' for very long context
TARGET_MODULES = "q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj"
# TARGET_MODULES = "q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj, embed_tokens, lm_head"
TARGET_MODULES = TARGET_MODULES.split(", ")


## Trainer Config
LR = 2e-4 #8e-6
LR_SCHEDULER_TYPE='cosine'
BETA = 0.1
EPOCHS = 1
GRADIENT_ACCUMULATION_STEPS = 4
MAX_STEPS = 500
WARMUP_STEPS = 100
WARMUP_RATIO = 0.1
LOGGING_STEPS = 1
OPTIMIZER = "paged_adamw_8bit"

# HF SAVING
HUB_STRATEGY = "all_checkpoints"
SAVING_STEPS = 100

DATASET_ID = 'Hrushi/SFT-MetaMath-40K-TrainingData-Part1'


## WANDB SETUP
wandb.login(key=os.getenv('WANDB_API_KEY'))
os.environ["WANDB_PROJECT"]="AIMO"
os.environ['WANDB_WATCH']='false'
os.environ["WANDB_LOG_MODEL"]='false'

wandb: Currently logged in as: hrushikesh. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/jupyter/.netrc


In [4]:
model, _ = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID, # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = MAX_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

tokenizer = AutoTokenizer.from_pretrained('gemma-7b-aimo-tokenizer')

==((====))==  Unsloth: Fast Gemma patching release 2024.5
   \\   /|    GPU: NVIDIA L4. Max memory: 21.951 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 8.9. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. Xformers = 0.0.26.post1. FA = True.
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = LORA_BIAS,
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = USE_GRADIENT_CHECKPOINTING, # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2024.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
    tokenizer.pad_token_id = tokenizer.unk_token_id

tokenizer.padding_side = 'right'

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    prompt     = examples["prompt"]
    completion = examples["completion"]
    text = f"{prompt}{completion}" + EOS_TOKEN

    return {'text': text}
pass

from datasets import load_dataset
# training_dataset = load_dataset(DATASET_FORMAT, data_files=DATASET_FPATH)['train']
training_dataset = load_dataset(DATASET_ID)['train']
training_dataset = training_dataset.map(formatting_prompts_func, batched = False)

RUN_NAME = f"{FT_MODEL_NAME}-N{training_dataset.num_rows}-E{EPOCHS}-R{LORA_R}-A{LORA_ALPHA}"
SAVE_FPATH = f"AIMO_Finetuned_Models/{RUN_NAME}"
OUTPUT_DIR = os.path.join('Training_Outputs', RUN_NAME)

In [7]:
training_dataset['text'][0]

"Your are a high school student appearing for your math exam.\nYou will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.\n\nFormat help and instructions:\n- Problem has been given enclosed in <math_problem></math_problem> tags.\n- Every step must be written within <start_of_step><end_of_step> tags.\n- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.\n- Code output will be given in <code_output_block></code_output_block> tags.\n\n<math_problem>In a classroom, $x$ children have $7$ books each. Their teacher brings another $8$ books to the classroom. How many books are in the classroom altogether?\nIf we know the answer to the above question is $78$, what is the value of unknown variable $x$?</math_problem>\n<start_of_solution>\n<start_of_step>1\nThe total number of books in 

In [8]:
tokenizer.tokenize(training_dataset['text'][0])

['Your',
 '▁are',
 '▁a',
 '▁high',
 '▁school',
 '▁student',
 '▁appearing',
 '▁for',
 '▁your',
 '▁math',
 '▁exam',
 '.',
 '\n',
 'You',
 '▁will',
 '▁be',
 '▁given',
 '▁a',
 '▁math',
 '▁problem',
 '▁to',
 '▁solve',
 ',',
 '▁and',
 '▁your',
 '▁job',
 '▁is',
 '▁to',
 '▁write',
 '▁the',
 '▁detailed',
 '▁step',
 '▁by',
 '▁step',
 '▁solution',
 ',',
 '▁with',
 '▁good',
 '▁mathematical',
 '▁reasoning',
 '▁and',
 '▁using',
 '▁python',
 '▁code',
 '▁for',
 '▁calculations',
 ',',
 '▁simpli',
 'fications',
 ',',
 '▁solving',
 '▁equations',
 ',',
 '▁etc',
 '.',
 '\n\n',
 'Format',
 '▁help',
 '▁and',
 '▁instructions',
 ':',
 '\n',
 '-',
 '▁Problem',
 '▁has',
 '▁been',
 '▁given',
 '▁enclosed',
 '▁in',
 '▁',
 '<math_problem>',
 '</math_problem>',
 '▁tags',
 '.',
 '\n',
 '-',
 '▁Every',
 '▁step',
 '▁must',
 '▁be',
 '▁written',
 '▁within',
 '▁',
 '<start_of_step>',
 '<end_of_step>',
 '▁tags',
 '.',
 '\n',
 '-',
 '▁Code',
 '▁needs',
 '▁to',
 '▁be',
 '▁always',
 '▁written',
 '▁inside',
 '▁',
 '<code_block>

In [9]:
response_template_ids = [9]

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = training_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_LENGTH,
    # max_prompt_length=MAX_PROMPT_LENGTH,
    data_collator=DataCollatorForCompletionOnlyLM(tokenizer=tokenizer, response_template=response_template_ids),
    formatting_func=formatting_prompts_func,
    dataset_num_proc = os.cpu_count(),
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        warmup_steps = WARMUP_STEPS,
        # warmup_ratio=WARMUP_RATIO,
        # max_steps = 60, # Set num_train_epochs = 1 for full training runs
        num_train_epochs=EPOCHS,
        learning_rate = LR,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = LOGGING_STEPS,
        optim = OPTIMIZER,
        lr_scheduler_type = LR_SCHEDULER_TYPE,
        seed = 3407,
        output_dir = OUTPUT_DIR,
        report_to='wandb',
        push_to_hub=True,
        resume_from_checkpoint=False,
        hub_strategy=HUB_STRATEGY,
        hub_private_repo=True,
        hub_model_id=RUN_NAME,
        run_name=RUN_NAME,
        save_strategy='steps',
        save_steps=SAVING_STEPS,
    ),
)

In [10]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 21.951 GB.
7.133 GB of memory reserved.


In [11]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 21,379 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 2,672
 "-____-"     Number of trainable parameters = 400,031,744


Step,Training Loss
1,2.326100
2,2.244900
3,2.457800
4,2.364500
5,2.296800
6,1.839600
7,2.218800
8,1.861600
9,1.640700
10,1.558500


In [12]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

19435.0095 seconds used for training.
323.92 minutes used for training.
Peak reserved memory = 16.439 GB.
Peak reserved memory for training = 9.306 GB.
Peak reserved memory % of max memory = 74.89 %.
Peak reserved memory for training % of max memory = 42.394 %.


In [15]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    """Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code></code> tags - No backticks or triple backticks.
- Code output will be given in <code_output></code_output> tags.

<math_problem>Write the predecessor of $10000$.</math_problem>
<start_of_solution>
"""
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048)

<bos>Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code></code> tags - No backticks or triple backticks.
- Code output will be given in <code_output></code_output> tags.

<math_problem>Write the predecessor of $10000$.</math_problem>
<start_of_solution>
<start_of_step>1
The predecessor of a number is the number that is one less than that number.
Therefore, the predecessor of $10000$ is $10000 - 1 =$<code_block>print(10000 - 1)</code_block>
<code_output_block>9999</code_output_block>
Hence, the predecessor of $10000$ is $\boxed{9999}$.<end_of_st

In [16]:
inputs = tokenizer(
[
    """Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code></code> tags - No backticks or triple backticks.
- Code output will be given in <code_output></code_output> tags.

<math_problem>Pick out the solution from the values given in the bracket next to each equation. Show that the other values do not satisfy the equation. $r - 4 = 0$ $(4, -4, 8, 0)$</math_problem>
<start_of_solution>
"""
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048)

<bos>Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code></code> tags - No backticks or triple backticks.
- Code output will be given in <code_output></code_output> tags.

<math_problem>Pick out the solution from the values given in the bracket next to each equation. Show that the other values do not satisfy the equation. $r - 4 = 0$ $(4, -4, 8, 0)$</math_problem>
<start_of_solution>
<start_of_step>1
To solve the given equation we will use `sympy`<code_block>from sympy import symbols, Eq, solve

# Define the variable
r = symbols('r')

# Define t

In [17]:
inputs = tokenizer(
[
    """Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code></code> tags - No backticks or triple backticks.
- Code output will be given in <code_output></code_output> tags.

<math_problem>If A and B are any two events such that P (A) + P (B) - P (A and B) = P (A), then (A) P (B|A) = 1 (B) P (A|B) = 1 (C) P (B|A) = 0 (D) P (A|B) = 0</math_problem>
<start_of_solution>
"""
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048, temperature=0)

<bos>Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code></code> tags - No backticks or triple backticks.
- Code output will be given in <code_output></code_output> tags.

<math_problem>If A and B are any two events such that P (A) + P (B) - P (A and B) = P (A), then (A) P (B|A) = 1 (B) P (A|B) = 1 (C) P (B|A) = 0 (D) P (A|B) = 0</math_problem>
<start_of_solution>
<start_of_step>1
We are given that P (A) + P (B) - P (A and B) = P (A). This means that the probability of A and B occurring together is the same as the probability of A occurring alon

In [18]:
# Push to HuggingFace Hub
commit_message = f"Finetuned {MODEL_ID} with SFT for {EPOCHS} epoch(s) on {training_dataset.num_rows} examples with r={LORA_R} and alpha={LORA_ALPHA}"
trainer.push_to_hub(commit_message=commit_message)

adapter_model.safetensors:   0%|          | 0.00/1.60G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Hrushi/AIMO-SFT-Unsloth-gemma-7b-bnb-4bit-AIMO-Tokenizer-N21379-E1-R128-A64/commit/d5ed426445a2358305a46c0921ff4bb934046fcb', commit_message='Finetuned unsloth/gemma-7b-bnb-4bit with SFT for 1 epoch(s) on 21379 examples with r=128 and alpha=64', commit_description='', oid='d5ed426445a2358305a46c0921ff4bb934046fcb', pr_url=None, pr_revision=None, pr_num=None)

In [19]:
print(commit_message)

Finetuned unsloth/gemma-7b-bnb-4bit with SFT for 1 epoch(s) on 21379 examples with r=128 and alpha=64
